## Importación de librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ne2_imp = pd.read_csv('../data_estaciones/BD_NE2_imputado_completo.csv', parse_dates=['time'])
ne3_imp = pd.read_csv('../data_estaciones/BD_NE3_imputado_completo.csv', parse_dates=['time'])
ne2_raw = pd.read_csv('../data_estaciones/BD_NE2_limpia.csv', parse_dates=['date']).rename(columns={'date':'time'})
ne3_raw = pd.read_csv('../data_estaciones/BD_NE3_limpia.csv', parse_dates=['date']).rename(columns={'date':'time'})
ndvi = pd.read_csv('../data_estaciones/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])

## Análisis en Meses de Mayor Riesgo 

In [3]:
# %% [markdown]
# # Análisis preliminar: vegetación, humedad y PM10 en temporada de riesgo
#
# **Pregunta:** ¿La vegetación (NDVI/EVI) influye en PM10 de forma indirecta,
# a través de la humedad relativa, específicamente durante la temporada de
# mayor riesgo sanitario (inversión térmica)?
#
# **Diseño:** en vez de comparar todos los meses del año (lo cual mezcla el
# ciclo estacional de vegetación con el ciclo estacional de contaminación y
# genera correlaciones espurias, como se documentó en el análisis anterior),
# se restringe la comparación a los mismos meses de mayor riesgo (dic-mar)
# a través de los 5 años disponibles. Así, cualquier variación remanente en
# NDVI/EVI es variación real año con año, no el ciclo anual.

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

DATA = '../data_estaciones'
MESES_RIESGO = [12, 1, 2, 3]  # confirmado con datos reales: pico de PM10 en ambas estaciones

# %% [markdown]
# ## Paso 1 — Confirmar la temporada de riesgo con datos propios
# (ya hecho antes, se reproduce aquí para que el notebook sea autocontenido)
for est in ['NE2', 'NE3']:
    df = pd.read_csv(f'{DATA}/BD_{est}_limpia.csv', parse_dates=['date'])
    df = df[(df.date >= '2021-01-01') & (df.date <= '2025-12-31')]
    m = df.set_index('date')['PM10'].resample('MS').median().reset_index()
    m['mesnum'] = m.date.dt.month
    print(f'{est} — PM10 promedio por mes:')
    print(m.groupby('mesnum').PM10.mean().round(1).to_dict())
# INTERPRETACIÓN: ambas estaciones muestran su pico dic-mar (inversión
# térmica, consistente con Cerón Bretón et al. 2020) y su mínimo ago-sep.
# Por eso se define MESES_RIESGO = [12,1,2,3].

# %% [markdown]
# ## Paso 2 — Construir la serie mensual restringida a temporada de riesgo
ndvi_mensual = pd.read_csv(f'{DATA}/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])
evi = pd.read_csv(f'{DATA}/EVI_estaciones.csv', parse_dates=['fecha'])

def construir_temporada_riesgo(estacion, indice='ndvi'):
    df = pd.read_csv(f'{DATA}/BD_{estacion}_limpia.csv', parse_dates=['date'])
    df = df[(df.date >= '2021-01-01') & (df.date <= '2025-12-31')]
    g = df.set_index('date')
    pm10 = g['PM10'].resample('MS').median()
    rh = g['RH'].resample('MS').mean()
    mensual = pd.DataFrame({'mes': pm10.index, 'PM10': pm10.values, 'RH': rh.values})

    if indice == 'ndvi':
        ind = ndvi_mensual[ndvi_mensual.estacion == estacion][['mes', 'ndvi_mediana']]
        ind = ind.rename(columns={'ndvi_mediana': 'indice_veg'})
    else:  # evi
        e = evi[evi.estacion == estacion].copy()
        e['mes'] = e.fecha.dt.to_period('M').dt.to_timestamp()
        ind = e.groupby('mes')['EVI'].mean().reset_index().rename(columns={'EVI': 'indice_veg'})

    d = mensual.merge(ind, on='mes').dropna().sort_values('mes').reset_index(drop=True)
    d['mesnum'] = d.mes.dt.month
    return d[d.mesnum.isin(MESES_RIESGO)].reset_index(drop=True)

# %% [markdown]
# ## Paso 3 — Resultados preliminares: correlaciones bivariadas
# (temporada de riesgo, 2021-2025, n=20 meses-año por estación)

resultados = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir_temporada_riesgo(est, indice)
        r_ind_pm10, p_ind_pm10 = pearsonr(d.indice_veg, d.PM10)
        r_ind_rh, p_ind_rh = pearsonr(d.indice_veg, d.RH)
        r_rh_pm10, p_rh_pm10 = pearsonr(d.RH, d.PM10)
        resultados.append({
            'indice': indice.upper(), 'estacion': est, 'n': len(d),
            'r(indice,PM10)': round(r_ind_pm10, 3), 'p(indice,PM10)': round(p_ind_pm10, 4),
            'r(indice,RH)': round(r_ind_rh, 3), 'p(indice,RH)': round(p_ind_rh, 4),
            'r(RH,PM10)': round(r_rh_pm10, 3), 'p(RH,PM10)': round(p_rh_pm10, 4),
        })
tabla_resultados = pd.DataFrame(resultados)
print(tabla_resultados.to_string(index=False))

NE2 — PM10 promedio por mes:
{1: 64.8, 2: 63.6, 3: 67.9, 4: 64.0, 5: 66.8, 6: 53.4, 7: 55.7, 8: 48.8, 9: 48.2, 10: 54.8, 11: 54.8, 12: 70.9}
NE3 — PM10 promedio por mes:
{1: 37.4, 2: 34.0, 3: 42.2, 4: 37.4, 5: 35.0, 6: 25.3, 7: 29.6, 8: 28.4, 9: 25.4, 10: 29.3, 11: 31.0, 12: 38.4}
indice estacion  n  r(indice,PM10)  p(indice,PM10)  r(indice,RH)  p(indice,RH)  r(RH,PM10)  p(RH,PM10)
  NDVI      NE2 20           0.498          0.0253        -0.140        0.5566      -0.169      0.4766
  NDVI      NE3 20           0.253          0.2815         0.342        0.1397      -0.349      0.1318
   EVI      NE2 20           0.435          0.0552         0.277        0.2362      -0.169      0.4766
   EVI      NE3 20           0.092          0.7006         0.600        0.0052      -0.349      0.1318


In [4]:
# %% [markdown]
# ## Paso 4 — Modelo multivariado: ¿el índice de vegetación aporta algo
# directo sobre PM10 una vez que se controla la humedad?
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir_temporada_riesgo(est, indice)
        X = sm.add_constant(d[['indice_veg', 'RH']])
        modelo = sm.OLS(d.PM10, X).fit()
        print(f'\n--- {indice.upper()} {est} ---')
        print(f'coef índice={modelo.params["indice_veg"]:.2f} (p={modelo.pvalues["indice_veg"]:.4f}) | '
              f'coef RH={modelo.params["RH"]:.3f} (p={modelo.pvalues["RH"]:.4f}) | R²={modelo.rsquared:.3f}')


--- NDVI NE2 ---
coef índice=503.89 (p=0.0347) | coef RH=-0.202 (p=0.6376) | R²=0.258

--- NDVI NE3 ---
coef índice=32.31 (p=0.0712) | coef RH=-0.405 (p=0.0380) | R²=0.279

--- EVI NE2 ---
coef índice=439.54 (p=0.0260) | coef RH=-0.625 (p=0.1612) | R²=0.280

--- EVI NE3 ---
coef índice=64.93 (p=0.0887) | coef RH=-0.517 (p=0.0268) | R²=0.263


## Utilizando todo el año

In [27]:
# %% [markdown]
# # Extensión: efectos fijos de mes para usar el año completo
# En vez de restringir el análisis a la temporada de riesgo (dic-mar), se
# usan TODOS los meses/semanas del año, pero se le agregan al modelo
# variables "dummy" (una por cada mes calendario) que absorben el promedio
# esperado de cada mes. Esto dejar solo la variación real (la desviación de
# cada mes/semana respecto a lo típico de esa época), sin necesitar
# restringir la muestra. Es matemáticamente equivalente a "deseasonalizar",
# pero hecho dentro de la misma regresión.

import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

DATA = '../data_estaciones'
ndvi_mensual = pd.read_csv(f'{DATA}/ndvi_mensual_estaciones_2020_2025.csv', parse_dates=['mes'])
evi = pd.read_csv(f'{DATA}/EVI_estaciones.csv', parse_dates=['fecha'])

# %% [markdown]
# ## Paso 1 — Construir las series mensuales (PM10, RH, índice de vegetación)
def construir_mensual(estacion, indice='ndvi'):
    """Agrega PM10 (mediana) y RH (promedio) a nivel mensual, y las une con
    el índice de vegetación elegido (NDVI o EVI)."""
    df = pd.read_csv(f'{DATA}/BD_{estacion}_limpia.csv', parse_dates=['date'])
    df = df[(df.date >= '2021-01-01') & (df.date <= '2025-12-31')]
    g = df.set_index('date')
    pm10 = g['PM10'].resample('MS').median()
    rh = g['RH'].resample('MS').mean()
    mensual = pd.DataFrame({'mes': pm10.index, 'PM10': pm10.values, 'RH': rh.values})

    if indice == 'ndvi':
        ind = ndvi_mensual[ndvi_mensual.estacion == estacion][['mes', 'ndvi_mediana']]
        ind = ind.rename(columns={'ndvi_mediana': 'indice_veg'})
    else:  # evi, compuesto de 16 dias -> se promedia a nivel mensual
        e = evi[evi.estacion == estacion].copy()
        e['mes'] = e.fecha.dt.to_period('M').dt.to_timestamp()
        ind = e.groupby('mes')['EVI'].mean().reset_index().rename(columns={'EVI': 'indice_veg'})

    d = mensual.merge(ind, on='mes').dropna().sort_values('mes').reset_index(drop=True)
    d['mesnum'] = d.mes.dt.month  # 1=enero, ..., 12=diciembre
    return d

# %% [markdown]
# ## Paso 2 — Función del modelo con efectos fijos de mes + errores robustos
def modelo_efectos_fijos(d, y_col, x_cols, maxlags=4):
    """
    Regresion OLS de y_col sobre x_cols, controlando el mes calendario con
    variables dummy (drop_first=True evita colinealidad perfecta: un mes
    queda como referencia implícita).

    Se usan errores estándar HAC (Newey-West) porque observaciones
    consecutivas en el tiempo suelen estar correlacionadas entre sí
    (autocorrelación); ignorarlo produce p-valores artificialmente
    pequeños. maxlags=4 para datos mensuales, se sube a 8 para semanales.
    """
    dummies = pd.get_dummies(d['mesnum'], prefix='m', drop_first=True).astype(float)
    X = pd.concat([d[x_cols].reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    modelo = sm.OLS(d[y_col].reset_index(drop=True), X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    return modelo

# %% [markdown]
# ## Paso 3 — Correr los tres modelos de la cadena causal propuesta:
# (a) índice de vegetación -> humedad relativa (RH)
# (b) RH -> PM10, controlando el índice de vegetación
# (c) índice de vegetación -> PM10, controlando RH (efecto directo residual)
resultados = []
for indice in ['ndvi', 'evi']:
    for est in ['NE2', 'NE3']:
        d = construir_mensual(est, indice)

        m_rh = modelo_efectos_fijos(d, 'RH', ['indice_veg'], maxlags=4)
        m_pm10 = modelo_efectos_fijos(d, 'PM10', ['indice_veg', 'RH'], maxlags=4)

        resultados.append({
            'indice': indice.upper(), 'estacion': est, 'n': len(d),
            'coef_indice_RH': round(m_rh.params['indice_veg'], 3),
            'p_indice_RH': round(m_rh.pvalues['indice_veg'], 4),
            'coef_RH_PM10': round(m_pm10.params['RH'], 3),
            'p_RH_PM10': round(m_pm10.pvalues['RH'], 4),
            'coef_indice_PM10': round(m_pm10.params['indice_veg'], 2),
            'p_indice_PM10': round(m_pm10.pvalues['indice_veg'], 4),
            'R2_modelo_PM10': round(m_pm10.rsquared, 3),
        })

tabla = pd.DataFrame(resultados)
print(tabla.to_string(index=False))

# %% [markdown]
# ### Interpretación del Paso 3 (mensual, n=60)
# - `coef_indice_RH` / `p_indice_RH`: si p<0.05, el índice de vegetación
#   predice humedad relativa de forma significativa, controlando el mes del
#   año. Un coeficiente positivo respalda el mecanismo de evapotranspiración
#   (más vegetación -> más humedad local).
# - `coef_RH_PM10` / `p_RH_PM10`: efecto de la humedad sobre PM10, ya
#   descontado el efecto del índice de vegetación. Se espera negativo
#   (más humedad favorece sedimentación de partículas).
# - `coef_indice_PM10` / `p_indice_PM10`: efecto DIRECTO del índice de
#   vegetación sobre PM10 que NO pasa por la humedad. Si este NO es
#   significativo pero el de RH sí, es evidencia de que el camino relevante
#   es indirecto (vegetación -> humedad -> PM10), no directo.
#
# RESULTADO REAL: RH->PM10 es significativo en las 4 combinaciones
# (p entre <0.001 y 0.035). EVI->RH es significativo en ambas estaciones
# (p=0.002 NE2, p<0.001 NE3) con signo positivo (esperado). NDVI->RH NO es
# significativo en ninguna estación (p=0.96, p=0.42) -- el vínculo con
# humedad solo aparece con EVI, no con NDVI.

# %% [markdown]
# ## Paso 4 — Repetir a nivel semanal (más observaciones, mismo control de mes)
def construir_semanal(estacion):
    """Usa el archivo semanal ya construido (PM10, NDVI, RH) sin valores nulos."""
    df = pd.read_csv(f'{DATA}/{estacion}_semanal_mediana_con_ndvi_sin_nulos.csv', parse_dates=['semana'])
    df = df[(df.semana >= '2021-01-01') & (df.semana <= '2025-12-31')]
    d = df[['semana', 'PM10', 'RH', 'ndvi_mediana']].dropna().reset_index(drop=True)
    d = d.rename(columns={'ndvi_mediana': 'indice_veg', 'semana': 'mes'})  # renombrar para reusar la funcion
    d['mesnum'] = d['mes'].dt.month
    return d

print("\n=== SEMANAL (n≈230, maxlags=8 por mayor autocorrelación semana a semana) ===")
for est in ['NE2', 'NE3']:
    d = construir_semanal(est)
    m_rh = modelo_efectos_fijos(d, 'RH', ['indice_veg'], maxlags=8)
    m_pm10 = modelo_efectos_fijos(d, 'PM10', ['indice_veg', 'RH'], maxlags=8)
    print(f'\n--- NDVI {est} semanal (n={len(d)}) ---')
    print(f'  NDVI -> RH:  coef={m_rh.params["indice_veg"]:.2f}, p={m_rh.pvalues["indice_veg"]:.4f}')
    print(f'  RH -> PM10:  coef={m_pm10.params["RH"]:.3f}, p={m_pm10.pvalues["RH"]:.4f}')
    print(f'  NDVI -> PM10: coef={m_pm10.params["indice_veg"]:.2f}, p={m_pm10.pvalues["indice_veg"]:.4f}')

# %% [markdown]
# ### Interpretación del Paso 4 (semanal)
# RESULTADO REAL: RH->PM10 sigue siendo muy significativo (p<0.001) en
# ambas estaciones -- confirma que es el hallazgo más robusto de todo el
# análisis, sin importar la resolución temporal.
#
# ADVERTENCIA: a nivel semanal, NDVI->RH SÍ sale significativo, pero con
# signo NEGATIVO (coef=-137 en NE2, -24 en NE3) -- contrario al positivo
# que dio EVI a nivel mensual. Esto NO debe interpretarse como un hallazgo
# real: es la misma inestabilidad de NDVI semanal (calculado con ~1.5
# imágenes satelitales por semana, muy ruidoso) que ya se había detectado
# antes en el proyecto con PM10 directo. Se recomienda reportar esto como
# limitación explícita, no como resultado a favor ni en contra.

# %% [markdown]
# ## Paso 5 — Resumen para el reporte (tabla final de confianza por hallazgo)
resumen_confianza = pd.DataFrame({
    'Hallazgo': [
        'RH -> PM10 (menor humedad, mayor PM10)',
        'EVI -> RH (más vegetación, más humedad) -- mensual',
        'NDVI -> RH -- mensual',
        'NDVI -> RH -- semanal',
        'Índice vegetación -> PM10 directo (controlando RH)'
    ],
    'Confianza': [
        'ALTA: 6/6 especificaciones significativas, mismo signo, robusto a HAC',
        'MODERADA: significativo en ambas estaciones, requiere más años para confirmar',
        'NULO: no significativo en ninguna estación',
        'NO CONFIABLE: significativo pero con signo contradictorio (ruido de medición)',
        'NULO/MARGINAL: no hay evidencia consistente de efecto directo'
    ]
})
print(resumen_confianza.to_string(index=False))

indice estacion  n  coef_indice_RH  p_indice_RH  coef_RH_PM10  p_RH_PM10  coef_indice_PM10  p_indice_PM10  R2_modelo_PM10
  NDVI      NE2 60          -3.276       0.9554        -0.567     0.0354            197.05         0.0792           0.458
  NDVI      NE3 60           5.083       0.4156        -0.471     0.0000              6.61         0.4760           0.531
   EVI      NE2 60         108.227       0.0016        -0.805     0.0002            182.58         0.0163           0.458
   EVI      NE3 60          63.046       0.0001        -0.609     0.0006             33.99         0.1414           0.549

=== SEMANAL (n≈230, maxlags=8 por mayor autocorrelación semana a semana) ===

--- NDVI NE2 semanal (n=231) ---
  NDVI -> RH:  coef=-136.76, p=0.0000
  RH -> PM10:  coef=-0.594, p=0.0000
  NDVI -> PM10: coef=57.14, p=0.1063

--- NDVI NE3 semanal (n=233) ---
  NDVI -> RH:  coef=-23.62, p=0.0004
  RH -> PM10:  coef=-0.575, p=0.0000
  NDVI -> PM10: coef=8.81, p=0.2158
                      